In [1]:
import os, re, csv, json, math, hashlib
from pathlib import Path
from typing import Dict, List, Tuple, Iterable, Optional

import pandas as pd

In [2]:
# -------- INPUTS --------
TRINEDAY_DIR = Path(r"C:\datasources\ai\trineday_mineru_txt_cleaned")
TRINEDAY_URLS_MAP = TRINEDAY_DIR / "trineday_urls_map.csv"  # filename,source_url

WTK_CLEAN_DIR = Path(r"C:\Users\fixin\OneDrive\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final")

SLIMMER_DIR = Path(r"C:\datasources\ai_corpus_slimmer\ai_corpus_slimmer_clean")
SLIMMER_PREFIXES = (
    "childrenshealthdefense.org",
    "franklin-cover-up",
    "Operation Paperclip",
    "usrtk.org",
)

WTK_CSV_PATH = Path(r"C:\datasources\260223WTK.csv")  # pipe-separated headers: ID|Source|Excerpt

# -------- OUTPUTS --------
OUT_DIR = Path(r"C:\datasources\ai\trineday_mini_build")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
def read_text_file(path: Path) -> str:
    # Robust decode: try utf-8 first, fallback to cp1252-ish.
    data = path.read_bytes()
    try:
        return data.decode("utf-8")
    except UnicodeDecodeError:
        return data.decode("utf-8", errors="replace")

def normalize_whitespace(s: str) -> str:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    # collapse excessive blank lines
    s = re.sub(r"\n{3,}", "\n\n", s)
    # collapse spaces/tabs
    s = re.sub(r"[ \t]{2,}", " ", s)
    return s.strip()

def sha1_short(s: str, n: int = 12) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()[:n]

def chunk_text_words(
    text: str,
    chunk_words: int = 480,
    overlap_words: int = 80,
    min_words: int = 80,
) -> List[str]:
    """
    Simple word-window chunker. Good enough for BM25 + embedding rerank.
    """
    words = text.split()
    if len(words) <= chunk_words:
        return [text.strip()] if len(words) >= min_words else []

    chunks = []
    step = max(1, chunk_words - overlap_words)
    for start in range(0, len(words), step):
        end = start + chunk_words
        window = words[start:end]
        if len(window) < min_words:
            break
        chunks.append(" ".join(window).strip())
        if end >= len(words):
            break
    return chunks

# Load Trine Day
def load_trineday_url_map(csv_path: Path) -> Dict[str, str]:
    m = {}
    with csv_path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            fn = (row.get("filename") or "").strip()
            url = (row.get("source_url") or "").strip()
            if fn:
                m[fn] = url
    return m

trineday_url_map = load_trineday_url_map(TRINEDAY_URLS_MAP)
len(trineday_url_map), list(trineday_url_map.items())[:3]

# Gather additional txt files
def list_txt_files(folder: Path) -> List[Path]:
    if not folder.exists():
        raise FileNotFoundError(folder)
    return sorted([p for p in folder.glob("*.txt") if p.is_file()])

# A) All Trineday mineru cleaned txt files (except the csv)
trineday_files = [p for p in list_txt_files(TRINEDAY_DIR) if p.name.lower() != "trineday_urls_map.csv"]

# B) All WTK clean_text_final txt files
wtk_clean_files = list_txt_files(WTK_CLEAN_DIR)

# C) Slimmer filtered files by prefix
slimmer_all = list_txt_files(SLIMMER_DIR)
slimmer_files = [
    p for p in slimmer_all
    if any(p.name.startswith(pref) for pref in SLIMMER_PREFIXES)
    and p.stat().st_size >= 4 * 1024
]

len(trineday_files), len(wtk_clean_files), len(slimmer_files)

(37, 1237, 1920)

In [8]:
from pathlib import Path

def domain_from_candidate(candidate: str) -> str:
    # candidate comes like "https://childrenshealthdefense.org/..."
    # we extract host in a simple way without urlparse overhead
    candidate = candidate.replace("http://", "").replace("https://", "")
    host = candidate.split("/", 1)[0]
    return host.lower().lstrip("www.")

def slimmer_filename_to_url_and_publisher(filename: str) -> tuple[str, str]:
    base = Path(filename).stem              # removes .txt
    candidate_path = base.replace("_", "/") # underscores -> slashes

    if not candidate_path.startswith(("http://", "https://")):
        candidate = "https://" + candidate_path
    else:
        candidate = candidate_path

    publisher = domain_from_candidate(candidate)

    # OFFLINE RULE:
    # Always use the constructed URL as source_url;
    # publisher is the domain.
    #
    # If you later want to "validate" links, do a separate notebook pass that
    # checks a sample, not inside chunk building.
    return candidate, publisher

In [9]:
from tqdm.auto import tqdm
import time

def build_txt_chunks(
    files,
    dataset_name: str,
    *,
    chunk_words: int = 480,
    overlap_words: int = 80,
    source_url_map=None,
    fixed_publisher=None,
    fixed_found_on=None,
    per_file_source_publisher_fn=None,
) -> List[dict]:
    rows = []
    t0 = time.perf_counter()

    for path in tqdm(files, desc=f"Chunking {dataset_name}", total=len(files)):
        raw = read_text_file(path)
        text = normalize_whitespace(raw)
        if not text:
            continue

        chunks = chunk_text_words(text, chunk_words=chunk_words, overlap_words=overlap_words)
        if not chunks:
            continue

        title = path.stem

        source_url = ""
        publisher = fixed_publisher or ""

        if per_file_source_publisher_fn is not None:
            source_url, publisher = per_file_source_publisher_fn(path.name)
        elif source_url_map is not None:
            source_url = source_url_map.get(path.name, "")

        found_on = fixed_found_on or ""

        for i, ch in enumerate(chunks):
            rows.append({
                "dataset": dataset_name,
                "filename": path.name,
                "source_url": source_url,
                "publisher": publisher,
                "found_on": found_on,
                "title": title,
                "chunk_index": i,
                "text": ch,
                "chunk_id": f"{dataset_name}:{path.name}:{i}:{sha1_short(ch)}",
            })

    print(f"{dataset_name}: {len(files)} files -> {len(rows)} chunks in {(time.perf_counter()-t0):.1f}s")
    return rows

# Trineday: chunked + mapped urls
rows_trineday = build_txt_chunks(
    trineday_files,
    dataset_name="trineday_mineru_txt_cleaned",
    source_url_map=trineday_url_map,
    fixed_publisher="Trine Day Publishing",
    fixed_found_on="Trine Day Publishing",
    chunk_words=480,
    overlap_words=80,
)

# WTK clean_text_final: chunked, no url map (you can add later if you have one)
rows_wtk_clean = build_txt_chunks(
    wtk_clean_files,
    dataset_name="wtk_clean_text_final",
    chunk_words=480,
    overlap_words=80,
)

# Slimmer subset: chunked, no url map (optional: derive url from filename prefix if you want)
rows_slimmer = build_txt_chunks(
    slimmer_files,
    dataset_name="ai_corpus_slimmer_clean_filtered",
    per_file_source_publisher_fn=slimmer_filename_to_url_and_publisher,
    chunk_words=480,
    overlap_words=80,
)

len(rows_trineday), len(rows_wtk_clean), len(rows_slimmer)

Chunking trineday_mineru_txt_cleaned:   0%|          | 0/37 [00:00<?, ?it/s]

trineday_mineru_txt_cleaned: 37 files -> 6602 chunks in 0.8s


Chunking wtk_clean_text_final:   0%|          | 0/1237 [00:00<?, ?it/s]

wtk_clean_text_final: 1237 files -> 9436 chunks in 1.3s


Chunking ai_corpus_slimmer_clean_filtered:   0%|          | 0/1920 [00:00<?, ?it/s]

ai_corpus_slimmer_clean_filtered: 1920 files -> 9417 chunks in 1.6s


(6602, 9436, 9417)

In [10]:
def load_wtk_pipe_csv(path: Path) -> List[dict]:
    df = pd.read_csv(path, sep="|", dtype=str, keep_default_na=False)

    out = []
    for _, r in df.iterrows():
        rid = (r.get("ID") or "").strip()
        src = (r.get("Source") or "").strip()
        excerpt = normalize_whitespace((r.get("Excerpt") or "").strip())
        if not excerpt:
            continue

        out.append({
            "dataset": "wtk_pipe_csv_260223",
            "filename": str(path.name),
            "source_url": src,
            "publisher": "WTK Archive",
            "found_on": "WTK Archive",
            "title": "",
            "chunk_index": 0,
            "text": excerpt,
            "chunk_id": f"wtkcsv:{rid}:{sha1_short(excerpt)}",
            "wtk_id": rid,
        })
    return out

rows_wtk_csv = load_wtk_pipe_csv(WTK_CSV_PATH)
len(rows_wtk_csv)

14143

In [11]:
all_rows = rows_trineday + rows_wtk_clean + rows_slimmer + rows_wtk_csv
len(all_rows)

def dedupe_rows(rows: List[dict]) -> List[dict]:
    seen = set()
    out = []
    for r in rows:
        key = sha1_short(r["text"], n=20)  # longer hash to reduce collision
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out

all_rows = dedupe_rows(all_rows)
len(all_rows)

df = pd.DataFrame(all_rows)

# Add a stable global index (this will become BM25 doc_idx)
df.insert(0, "doc_idx", range(len(df)))

out_jsonl = OUT_DIR / "trineday_mini_chunks.jsonl"
df.to_json(out_jsonl, orient="records", lines=True, force_ascii=False)

out_csv = OUT_DIR / "trineday_mini_chunks.csv"
df.to_csv(out_csv, index=False)

out_jsonl, out_csv, df.shape

(WindowsPath('C:/datasources/ai/trineday_mini_build/trineday_mini_chunks.jsonl'),
 WindowsPath('C:/datasources/ai/trineday_mini_build/trineday_mini_chunks.csv'),
 (38446, 11))

In [12]:
import math

N = len(df)  # 38446
TARGET_SHARDS = 64  
SHARD_SIZE = math.ceil(N / TARGET_SHARDS)
SHARD_SIZE

601

In [13]:
import json, math
from pathlib import Path
import pandas as pd
import bm25s

IN_JSONL = Path(r"C:\datasources\ai\trineday_mini_build\trineday_mini_chunks.jsonl")
OUT_DIR = Path(r"C:\datasources\ai\trineday_mini_build")
BM25_DIR = OUT_DIR / "bm25_index"
SHARDS_DIR = OUT_DIR / "groups"

BM25_DIR.mkdir(parents=True, exist_ok=True)
SHARDS_DIR.mkdir(parents=True, exist_ok=True)

# Load jsonl
df = pd.read_json(IN_JSONL, lines=True)

# Ensure doc_idx exists and is contiguous 0..N-1
df = df.sort_values("doc_idx").reset_index(drop=True)
df["doc_idx"] = range(len(df))

N = len(df)
print("Docs:", N)

resource module not available on Windows
Docs: 38446


In [14]:
# Corpus text used for BM25 scoring
# You can tune fielding (title boost) later if you want:
# corpus = (df["title"].fillna("") + "\n" + df["title"].fillna("") + "\n" + df["text"].fillna("")).tolist()
corpus = (df["title"].fillna("") + "\n" + df["text"].fillna("")).tolist()

tokens = bm25s.tokenize(corpus, stopwords="en")  # no stemming
bm25 = bm25s.BM25()
bm25.index(tokens)

bm25.save(str(BM25_DIR))
print("Saved bm25 index to:", BM25_DIR)

Split strings:   0%|          | 0/38446 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/38446 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/38446 [00:00<?, ?it/s]

Saved bm25 index to: C:\datasources\ai\trineday_mini_build\bm25_index


In [15]:
TARGET_SHARDS = 64  # choose between 100 and 200
SHARD_SIZE = math.ceil(N / TARGET_SHARDS)
print("Shard size:", SHARD_SIZE, "Num shards:", math.ceil(N / SHARD_SIZE))

Shard size: 601 Num shards: 64


In [16]:
def write_group_shards(df: pd.DataFrame, shards_dir: Path, shard_size: int) -> dict:
    n = len(df)
    num_shards = math.ceil(n / shard_size)

    for sid in range(num_shards):
        start = sid * shard_size
        end = min(n, start + shard_size)

        shard_rows = df.iloc[start:end].to_dict(orient="records")
        out_path = shards_dir / f"group_{sid:04d}.json"
        out_path.write_text(json.dumps(shard_rows, ensure_ascii=False), encoding="utf-8")

    meta = {
        "num_docs": n,
        "shard_size": shard_size,
        "num_shards": num_shards,
        "group_naming": "group_{sid:04d}.json",
    }
    (shards_dir / "shards_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return meta

meta = write_group_shards(df, SHARDS_DIR, SHARD_SIZE)
meta

{'num_docs': 38446,
 'shard_size': 601,
 'num_shards': 64,
 'group_naming': 'group_{sid:04d}.json'}

In [3]:
# Local BM25S smoke test + top-10 search on Trine Day Mini
# Assumes you already created:
#   C:\datasources\ai\trineday_mini_build\trineday_mini_chunks.jsonl
# and (optional) a saved bm25 index at:
#   C:\datasources\ai\trineday_mini_build\bm25_index
#
# This cell will:
#  1) load the dataset
#  2) build (or load) a bm25 index
#  3) run a query
#  4) print top-10 results with doc_idx, publisher, found_on, source_url, title, excerpt preview, and score

import os, time
from pathlib import Path
import pandas as pd
import numpy as np
import bm25s

# ---------- CONFIG ----------
DATA_JSONL = Path(r"C:\datasources\ai\trineday_mini_build\trineday_mini_chunks.jsonl")
BM25_DIR   = Path(r"C:\datasources\ai\trineday_mini_build\bm25_index")
TOP_K      = 10

# If you want title-weighting, set TITLE_BOOST=2 or 3
TITLE_BOOST = 1  # 1 = no boost, 2 = title repeated twice, etc.

# ---------- LOAD DATA ----------
t0 = time.perf_counter()
df = pd.read_json(DATA_JSONL, lines=True)

# Ensure doc_idx is contiguous and sorted
if "doc_idx" not in df.columns:
    df.insert(0, "doc_idx", range(len(df)))
df = df.sort_values("doc_idx").reset_index(drop=True)
df["doc_idx"] = range(len(df))

print(f"Loaded {len(df):,} rows in {(time.perf_counter()-t0):.2f}s")

# ---------- BUILD OR LOAD BM25 ----------
def build_corpus(dframe: pd.DataFrame, title_boost: int = 1):
    title = dframe["title"].fillna("").astype(str)
    text  = dframe["text"].fillna("").astype(str)
    if title_boost <= 1:
        return (title + "\n" + text).tolist()
    boosted_title = ("\n".join([title.name]*0))  # no-op, kept for clarity
    # Repeat title_boost times by concatenating strings:
    # title + "\n" + title + "\n" + ... + "\n" + text
    rep = title
    for _ in range(title_boost - 1):
        rep = rep + "\n" + title
    return (rep + "\n" + text).tolist()

def load_or_build_bm25():
    # Try load first if the directory looks like a saved index
    if BM25_DIR.exists() and any(BM25_DIR.iterdir()):
        try:
            t = time.perf_counter()
            bm25 = bm25s.BM25.load(str(BM25_DIR), load_corpus=False)
            print(f"Loaded BM25 index from disk in {(time.perf_counter()-t):.2f}s  ({BM25_DIR})")
            return bm25, False
        except Exception as e:
            print("Could not load BM25 index; will rebuild. Reason:", repr(e))

    # Build
    t = time.perf_counter()
    corpus = build_corpus(df, title_boost=TITLE_BOOST)
    tokens = bm25s.tokenize(corpus, stopwords="en")  # no stemming
    bm25 = bm25s.BM25()
    bm25.index(tokens)
    print(f"Built BM25 index in {(time.perf_counter()-t):.2f}s")

    # Save for next time
    BM25_DIR.mkdir(parents=True, exist_ok=True)
    bm25.save(str(BM25_DIR))
    print(f"Saved BM25 index to: {BM25_DIR}")
    return bm25, True

bm25, rebuilt = load_or_build_bm25()

# ---------- SEARCH ----------
def bm25_search(query: str, k: int = 10):
    t = time.perf_counter()
    q_tokens = bm25s.tokenize(query, stopwords="en")
    doc_ids, scores = bm25.retrieve(q_tokens, k=k)
    elapsed = time.perf_counter() - t

    # doc_ids/scores are typically shape (1, k)
    doc_ids = doc_ids[0]
    scores = scores[0]

    # doc_ids should be integer indices aligned to df.doc_idx
    # If they are strings (rare depending on how bm25s is loaded), we handle that too.
    results = []
    for rank, (doc_id, score) in enumerate(zip(doc_ids, scores), start=1):
        # Convert numpy scalars to python
        if hasattr(doc_id, "item"):
            doc_id = doc_id.item()
        if hasattr(score, "item"):
            score = score.item()

        if isinstance(doc_id, (int, np.integer)):
            row = df.iloc[int(doc_id)]
        else:
            # Fallback: if bm25 returns the doc text itself (when corpus was loaded),
            # we locate it (slow). If this happens, tell me and we'll adjust your load.
            doc_text = str(doc_id)
            # naive lookup (slow but OK for a smoke test)
            idx = df.index[(df["title"].fillna("") + "\n" + df["text"].fillna("")).astype(str) == doc_text]
            row = df.iloc[int(idx[0])] if len(idx) else None

        if row is None:
            continue

        txt = str(row.get("text",""))
        excerpt = (txt[:350] + "…") if len(txt) > 350 else txt

        results.append({
            "rank": rank,
            "score": float(score),
            "doc_idx": int(row["doc_idx"]),
            "publisher": row.get("publisher",""),
            "found_on": row.get("found_on",""),
            "source_url": row.get("source_url",""),
            "title": row.get("title",""),
            "excerpt": excerpt,
        })

    return results, elapsed

query = "who killed Gandhi"  # change me
results, elapsed = bm25_search(query, k=TOP_K)

print(f"\nQuery: {query!r}")
print(f"Top {TOP_K} in {elapsed*1000:.1f} ms\n")

for r in results:
    print(f'#{r["rank"]} score={r["score"]:.4f} doc_idx={r["doc_idx"]} | {r["publisher"]} | {r["found_on"]}')
    print(r["title"])
    print(r["source_url"])
    print(r["excerpt"])
    print("-" * 80)

Loaded 38,446 rows in 1.55s
Loaded BM25 index from disk in 0.11s  (C:\datasources\ai\trineday_mini_build\bm25_index)


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Query: 'who killed Gandhi'
Top 10 in 15.0 ms

#1 score=5.2899 doc_idx=8648 |  | 
9909intelreport

the LTTE's elite commando unit known as the Black Tigers. Members of this unit are designated as "suicide commandos" and carry around their necks a glass vial containing potassium cyanide. Suicide is common in Hindu society, and the Tigers are fanatical Hindus. The cyanide capsule, which LTTE members view as the ultimate symbol of bravery and commi…
--------------------------------------------------------------------------------
#2 score=4.9834 doc_idx=35399 | WTK Archive | WTK Archive

https://www.wanttoknow.info/a-pandemic-exit-interviews-stop-panicking-about-covid19-variants-says-ucsfs-monica-gandhi
Pandemic Exit Interviews: Stop panicking about the COVID-19 variants, says UCSF's Monica Gandhi as reported by San Francisco Chronicle (San Francisco's leading newspaper) Dr. Monica Gandhi is not your typical epidemiologist in the era of COVID-19. While the vast majority of experts in her f